In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
cols = ['Sex', 'Length', 'Diameter', 'Height', 'Whole_w', 'Shucked_w', 'Viscera_w', 'Shell_w', 'Rings']
df = pd.read_csv(url, names=cols)

# 2. Preprocessing

encoder = OneHotEncoder(sparse_output=False)
sex_encoded = encoder.fit_transform(df[['Sex']])
X = np.hstack((sex_encoded, df.drop(['Sex', 'Rings'], axis=1).values))
y = df['Rings'].values.reshape(-1, 1)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. MLP with Backpropagation from scratch
class MLP:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.w1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.w2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        self.lr = lr

    def forward(self, X):
        self.z1 = np.dot(X, self.w1) + self.b1
        self.a1 = np.tanh(self.z1)
        self.z2 = np.dot(self.a1, self.w2) + self.b2
        return self.z2 # Output (Linear for regression)

    def backward(self, X, y, output):
        m = y.shape[0]
        dz2 = output - y
        dw2 = np.dot(self.a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m

        da1 = np.dot(dz2, self.w2.T)
        dz1 = da1 * (1 - np.power(self.a1, 2))
        dw1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m

        # Update weights
        self.w1 -= self.lr * dw1
        self.b1 -= self.lr * db1
        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2

    def train(self, X, y, epochs=1000):
        for i in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output)
            if i % 200 == 0:
                mse = np.mean((output - y)**2)
                print(f"Epoch {i}, Loss (MSE): {mse:.4f}")

# 4. Train
model = MLP(input_size=X_train.shape[1], hidden_size=10, output_size=1, lr=0.1)
model.train(X_train, y_train, epochs=1000)

# 5. Evaluate
predictions = model.forward(X_test)
print(f"\nFinal RMSE: {np.sqrt(np.mean((predictions - y_test)**2)):.4f}")

Epoch 0, Loss (MSE): 109.1876
Epoch 200, Loss (MSE): 4.5037
Epoch 400, Loss (MSE): 4.3672
Epoch 600, Loss (MSE): 4.3198
Epoch 800, Loss (MSE): 4.2865

Final RMSE: 2.1631
